# P&ID Safety Copilot
**Gemma 4 Good Hackathon** | Theme: Climate & Environmental Impact

Fine-tuned Gemma 4 E4B that answers safety-critical questions about Piping and
Instrumentation Diagrams (P&IDs). Given a P&ID component graph, the model identifies
isolation valves, lists components by type, and flags single-point failure risks.

Runs fully offline on consumer hardware (T4 GPU). No cloud dependency.


In [ ]:
# Install dependencies
!pip install unsloth[colab-new] trl datasets networkx -q
!git clone https://github.com/govindrathore27/gemma4-engineering-diagrams.git /kaggle/working/repo 2>/dev/null || git -C /kaggle/working/repo pull
import sys
sys.path.insert(0, "/kaggle/working/repo")
print("Setup complete")


In [ ]:
# Train the P&ID Safety Copilot model
import os
os.environ["WANDB_DISABLED"] = "true"

from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="pid",
    data_path="/kaggle/input/gemma4-pid-train-data/train.jsonl",
    output_dir="/kaggle/working/pid_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")


In [ ]:
# Evaluate on held-out eval set
import json
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/pid_adapter")
print("Model loaded")

eval_rows = [json.loads(l) for l in open("/kaggle/input/gemma4-pid-train-data/eval.jsonl", encoding="utf-8")]
# Use first 100 for speed
eval_rows = eval_rows[:100]

predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected = [r["output"] for r in eval_rows]

result = evaluate_batch(predictions, expected, metric="f1")
print(f"Token F1: {result['mean']:.3f}")

# Save eval results
with open("/kaggle/working/eval_results.txt", "w") as f:
    f.write(f"Token F1: {result['mean']:.3f}\n")
    f.write(f"Evaluated on {len(eval_rows)} samples\n")


In [ ]:
# Export to GGUF for offline deployment
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/pid_adapter",
    output_path="/kaggle/working/pid_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")


In [ ]:
# Demo inference - P&ID Safety Q&A
sample_graph = """
{
  "nodes": [
    {"id": "pump1", "label": "pump"},
    {"id": "valve1", "label": "valve"},
    {"id": "valve2", "label": "valve"},
    {"id": "valve3", "label": "valve"},
    {"id": "vessel1", "label": "vessel"},
    {"id": "instrumentation1", "label": "instrumentation"}
  ],
  "edges": [
    {"from": "pump1", "to": "valve1"},
    {"from": "valve1", "to": "vessel1"},
    {"from": "valve2", "to": "vessel1"},
    {"from": "valve3", "to": "vessel1"},
    {"from": "instrumentation1", "to": "vessel1"}
  ]
}
"""

questions = [
    "List all valve components in this P&ID.",
    "What valves must be closed to isolate pump1?",
    "List all instrumentation components in this P&ID.",
    "How many components of each type are in this P&ID?",
    "Identify single-point failures that could affect vessel1.",
]

print("=== P&ID Safety Copilot Demo ===\n")
for q in questions:
    answer = run(model, tokenizer, q, sample_graph)
    print(f"Q: {q}")
    print(f"A: {answer}\n")


## Real-World Impact

Process safety engineers spend hours manually tracing P&IDs to answer isolation
questions. This model enables instant, auditable answers — reducing accident risk
in chemical plants, water treatment facilities, and nuclear installations.

**Key capabilities:**
- Identifies isolation valve paths for any component
- Lists components by type (valves, instruments, pumps, etc.)
- Flags single-point failure risks for vessels
- Answers natural language safety queries

**Deployment:** Runs fully offline on a T4 GPU (15 GB VRAM). GGUF export enables
CPU-only inference via llama.cpp with no internet connection required.

**Datasets used:**
- PID2Graph (Zenodo, CC BY-SA): 500 annotated P&IDs with graph structure
- Training: 3,986 auto-generated QA pairs from graph traversals
